# Graph Intelligence: Neo4j GraphRAG + OpenAI Agents SDK

[`neo4j-graphrag`](https://neo4j.com/docs/neo4j-graphrag-python/current/) is Neo4j's official
retrieval library. It provides **retrievers** - objects that turn a question into results from your
graph using vector search, full-text search, graph traversal, or a combination of all three.

This notebook wires those retrievers into the [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/):

1. **Retrievers as tools** - each retrieval strategy wrapped with `@function_tool`, chosen by one agent
2. **Specialist agents and handoffs** - a triage agent that transfers control to a retrieval specialist

The demo uses the publicly accessible **Neo4j companies knowledge graph** (`Article` nodes split into
`Chunk` nodes, linked to the `Organization` nodes they mention).

## 1. Setup

Install the required Python packages.

In [1]:
!pip install --quiet --upgrade openai-agents neo4j-graphrag neo4j
print("Packages installed ✓")


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Packages installed ✓


## 2. Configuration

Set environment variables for Neo4j and OpenAI. The **companies demo database** is public and
read-only, so no signup is needed. The same OpenAI key covers both the agent model and the
embeddings.

In [2]:
import json
import math
import os
from getpass import getpass

import neo4j
from neo4j import GraphDatabase

from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.retrievers import (
    VectorRetriever,
    VectorCypherRetriever,
    HybridRetriever,
    HybridCypherRetriever,
)
from neo4j_graphrag.types import RetrieverResultItem

from agents import Agent, Runner, function_tool

# OpenAI API key
os.environ.setdefault("OPENAI_API_KEY", getpass("OpenAI API key: "))

MODEL = "gpt-5.4-mini"

# Neo4j Knowledge Graph - read-only companies demo (public credentials)
os.environ["NEO4J_URI"]      = "neo4j+s://demo.neo4jlabs.com:7687"
os.environ["NEO4J_USERNAME"] = "companies"
os.environ["NEO4J_PASSWORD"] = "companies"
os.environ["NEO4J_DATABASE"] = "companies"

driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
)

print("Configuration set.")
print("Neo4j URI: ", os.environ["NEO4J_URI"])
print("Model:     ", MODEL)

Configuration set.
Neo4j URI:  neo4j+s://demo.neo4jlabs.com:7687
Model:      gpt-5.4-mini


### The graph

Articles are split into `Chunk` nodes carrying the text and its embeddings, and linked to the
organizations they mention:

```
(Article)-[:HAS_CHUNK]->(Chunk)
(Article)-[:MENTIONS]->(Organization)
(Organization)-[:HAS_COMPETITOR|HAS_INVESTOR|HAS_SUPPLIER]->(Organization)
```

That shape is what makes graph retrieval worthwhile. A vector search finds a *chunk* - but the
article it belongs to, its date, its sentiment, and the companies it discusses are all one hop away.

### Cypher version

`neo4j-graphrag` generates its vector searches using the Cypher 25 `SEARCH` clause. The demo database
understands Cypher 25 but still defaults to Cypher 5, so we prefix the generated queries to select
the newer language version.

Skip this cell if your own database already defaults to Cypher 25.

In [3]:
if not getattr(neo4j.Driver.execute_query, "_cypher25_patch", False):
    _execute_query = neo4j.Driver.execute_query

    def _execute_query_cypher25(self, query, *args, **kwargs):
        if isinstance(query, str) and "VECTOR INDEX" in query.upper() \
                and not query.lstrip().upper().startswith("CYPHER"):
            query = "CYPHER 25 " + query
        return _execute_query(self, query, *args, **kwargs)

    _execute_query_cypher25._cypher25_patch = True
    neo4j.Driver.execute_query = _execute_query_cypher25

print("Generated vector queries will run as Cypher 25 ✓")

Generated vector queries will run as Cypher 25 ✓


## 3. Pair the embedding model with the index

A vector index stores embeddings produced by one specific model. Querying it with a different model
still returns results, but they are meaningless: two models place the same text at different points
in space, so the similarity scores are noise.

Nothing raises an exception when this happens, which makes it the most common way a GraphRAG setup
silently underperforms. So start by looking at what indexes exist on `Chunk`.

In [4]:
indexes, _, _ = driver.execute_query(
    """
    SHOW INDEXES YIELD name, type, labelsOrTypes, properties, options
    WHERE type IN ['VECTOR', 'FULLTEXT'] AND labelsOrTypes = ['Chunk']
    RETURN name, type, properties,
           options.indexConfig['vector.dimensions'] AS dimensions
    ORDER BY type, name
    """,
    database_=os.environ["NEO4J_DATABASE"],
)

for record in indexes:
    row = record.data()
    dims = f"{row['dimensions']} dims" if row["dimensions"] else "-"
    print(f"  {row['name']:<18} {row['type']:<9} {str(row['properties']):<26} {dims}")

  news_fulltext      FULLTEXT  ['text']                   -
  news               VECTOR    ['embedding']              1536 dims
  news_google        VECTOR    ['embedding_google']       768 dims
  news_google_004    VECTOR    ['embedding_google_004']   768 dims
  news_sbert         VECTOR    ['embedding_sbert']        384 dims


The demo graph stores several embeddings of the same chunks, one per model, so you can use whichever
provider you already have access to.

We use **`news`**, whose 1536 dimensions correspond to OpenAI's `text-embedding-ada-002` - the
default model for `neo4j-graphrag`'s `OpenAIEmbeddings`. Since we are already using OpenAI for the
agent, no extra credentials are needed.

**`news_fulltext`** indexes the `text` property on those same `Chunk` nodes. Hybrid retrieval needs a
vector index and a full-text index over the same nodes, which is what makes section 6 possible.

To confirm the pairing, re-embed a chunk that already has a stored vector and compare the two. An
exact model match scores close to 1.0; anything much lower means the index was built with a different
model, and you should pick another index or another embedder.

In [5]:
VECTOR_INDEX = "news"
FULLTEXT_INDEX = "news_fulltext"

embedder = OpenAIEmbeddings()


def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))


sample, _, _ = driver.execute_query(
    "MATCH (c:Chunk) WHERE c.embedding IS NOT NULL "
    "RETURN c.text AS text, c.embedding AS stored LIMIT 1",
    database_=os.environ["NEO4J_DATABASE"],
)

score = cosine(embedder.embed_query(sample[0]["text"]), sample[0]["stored"])
print(f"Self-similarity: {score:.3f}",
      "- models match ✓" if score > 0.95 else "- MISMATCH, pick a different index or embedder")

Self-similarity: 1.000 - models match ✓


## 4. VectorRetriever - semantic search

The simplest retriever. Give it a driver, an index name, and an embedder; it embeds the question,
searches the index, and returns matching nodes.

Set `return_properties` to control what comes back. Without it you get whole nodes - embeddings
included, which is thousands of floats per result heading straight into your prompt.

In [7]:
vector_retriever = VectorRetriever(
    driver=driver,
    index_name=VECTOR_INDEX,
    embedder=embedder,
    return_properties=["text"],
    neo4j_database=os.environ["NEO4J_DATABASE"],
)

QUERY = "data centres trendsnt"

for item in vector_retriever.search(query_text=QUERY, top_k=3).items:
    print(f"  {item.content[:160]}...\n")

  {'text': 'NTT Ltd. opened its first data center in Spain.\nThe high-availability, Tier 3-compliant colocation data center is located on NTT\'s Európolis Busines...

  {'text': "NTT, a global leader in technology services, has joined forces with Prestige Group to launch a state-of-the-art data center in Bengaluru, marking Pres...

  {'text': 'The new facility in Madrid offers 6.3 MW of IT capacity and excellent connectivity to many carriers and cloud providers\nHATTERSHEIM, Germany & MADRID...



This is ordinary RAG: it finds relevant text and stops. The retriever knows a chunk matched, but not
which article it came from, when it was published, or which companies it discusses.

## 5. VectorCypherRetriever - retrieval that continues into the graph

This is the retriever that separates GraphRAG from RAG.

It runs the same vector search, then executes a `retrieval_query` starting from each match. Two
variables are in scope for that query: `node`, the node found by the vector search, and `score`, its
similarity. From there you can traverse anywhere in the graph.

Below: chunk → article (title, date, sentiment) → organizations mentioned → their competitors. The
model receives the text *and* the surrounding facts in a single round trip.

A `result_formatter` shapes each returned record. Without one you get the driver's default record
repr, which is awkward to read and awkward to hand to a model.

In [8]:
RETRIEVAL_QUERY = """
WITH node AS chunk, score
MATCH (article:Article)-[:HAS_CHUNK]->(chunk)
OPTIONAL MATCH (article)-[:MENTIONS]->(org:Organization)
OPTIONAL MATCH (org)-[:HAS_COMPETITOR]->(rival:Organization)
RETURN chunk.text             AS text,
       article.id             AS article_id,
       article.title          AS title,
       toString(article.date) AS date,
       article.sentiment      AS sentiment,
       collect(DISTINCT org.name)[..5]   AS companies,
       collect(DISTINCT rival.name)[..5] AS competitors,
       score
ORDER BY score DESC
"""


def format_record(record) -> RetrieverResultItem:
    """Shape each result into compact JSON for the model."""
    return RetrieverResultItem(
        content=json.dumps({
            "text": (record.get("text") or "")[:600],
            "article_id": record.get("article_id"),
            "title": record.get("title"),
            "date": record.get("date"),
            "sentiment": record.get("sentiment"),
            "companies": record.get("companies"),
            "competitors": record.get("competitors"),
        }, default=str),
        metadata={"score": record.get("score")},
    )


graph_retriever = VectorCypherRetriever(
    driver=driver,
    index_name=VECTOR_INDEX,
    retrieval_query=RETRIEVAL_QUERY,
    embedder=embedder,
    result_formatter=format_record,
    neo4j_database=os.environ["NEO4J_DATABASE"],
)

for item in graph_retriever.search(query_text=QUERY, top_k=2).items:
    row = json.loads(item.content)
    print(f"  {row['title']}  ({row['date']}, sentiment {row['sentiment']})")
    print(f"    companies:   {row['companies']}")
    print(f"    competitors: {row['competitors']}")
    print(f"    {row['text'][:120]}...\n")

  NTT inaugurates data center in Madrid  (2022-05-24T00:00:00Z, sentiment 0.87)
    companies:   ['NetIX', 'Google', 'Microsoft']
    competitors: []
    NTT Ltd. opened its first data center in Spain.
The high-availability, Tier 3-compliant colocation data center is locate...

  NTT and Prestige Group Collaborate to Establish a 100MW Data Center in Bengaluru  (2023-06-13T05:37:29Z, sentiment 0.91)
    companies:   ['Nippon Telegraph and Telephone']
    competitors: ['KDDI', 'Sify', 'Transatel', 'China Mobile', 'Bharti Airtel']
    NTT, a global leader in technology services, has joined forces with Prestige Group to launch a state-of-the-art data cen...



Same query and same index as section 4, but every result now carries an article ID, a date, a stored
sentiment score, and the companies involved. That buys three things plain vector search cannot offer:

- **Citations you can check** - `article_id` is a real node, so a claim can be traced back to its source
- **Facts the text never states** - a competitor relationship lives in the graph, not in the article body
- **Structure to filter and rank on** - date, sentiment, and relationships are all available after the semantic match

## 6. HybridRetriever - semantic plus keyword

Vector search is strong on meaning and weak on exact strings. Ask about a specific company, ticker, or
product and it tends to return material on the right *topic* while missing the document that names it
outright. Full-text search has the opposite profile: exact terms, no understanding of meaning.

`HybridRetriever` runs both and merges the rankings, which recovers the exact matches without losing
the semantic ones.

In [9]:
hybrid_retriever = HybridRetriever(
    driver=driver,
    vector_index_name=VECTOR_INDEX,
    fulltext_index_name=FULLTEXT_INDEX,
    embedder=embedder,
    return_properties=["text"],
    neo4j_database=os.environ["NEO4J_DATABASE"],
)

NAMED_QUERY = "Nvidia data center GPU demand"

print("--- vector only ---")
for item in vector_retriever.search(query_text=NAMED_QUERY, top_k=3).items:
    print(f"  {item.content[:120]}...")

print("\n--- hybrid ---")
for item in hybrid_retriever.search(query_text=NAMED_QUERY, top_k=3).items:
    print(f"  {item.content[:120]}...")

--- vector only ---
  {'text': '. Traditional GPU uses include professional visualization applications that require realistic rendering, inclu...
  {'text': '. Hyperscale cloud vendors have leveraged GPUs in training neural networks for uses such as image and speech r...
  {'text': '(Reuters) - Nvidia Corp <NVDA.O> on Monday laid out a multi-year plan to create a new kind of chip for data ce...

--- hybrid ---
  {'text': 'Supermicro Founder and CEO Charles Liang Will be Joined by Jensen Huang, NVIDIA CEO, and other Industry Lumina...
  {'text': '. Traditional GPU uses include professional visualization applications that require realistic rendering, inclu...
  {'text': 'Supermicro Founder and CEO Charles Liang Will be Joined by Jensen Huang, NVIDIA CEO, and other Industry Lumina...


The hybrid results surface chunks naming the company directly, while vector-only results drift toward
the general theme. The more distinctive the term - proper nouns, tickers, product codes - the wider
the gap.

`HybridCypherRetriever` combines both ideas: hybrid search plus a `retrieval_query`. It is the one to
reach for in practice, and the one we give the agent below.

In [10]:
hybrid_graph_retriever = HybridCypherRetriever(
    driver=driver,
    vector_index_name=VECTOR_INDEX,
    fulltext_index_name=FULLTEXT_INDEX,
    retrieval_query=RETRIEVAL_QUERY,
    embedder=embedder,
    result_formatter=format_record,
    neo4j_database=os.environ["NEO4J_DATABASE"],
)

row = json.loads(hybrid_graph_retriever.search(query_text=NAMED_QUERY, top_k=1).items[0].content)
print(json.dumps(row, indent=2)[:600])

{
  "text": ". Traditional GPU uses include professional visualization applications that require realistic rendering, including computer-aided design, video editing, and special effects. Nvidia has experienced success in focusing its GPUs in burgeoning markets such as artificial intelligence (deep learning) and self-driving vehicles. Hyperscale cloud vendors have leveraged GPUs in training neural networks for uses such as image and speech recognition, large language models (ChatGPT), and other forms of generative AI.\u201d\n\u2014Brian Colello, director of technology equity research\n\u201cWe 


## 7. Retrievers as function tools

A retriever is a Python object with a `search()` method, so exposing one to an agent means wrapping it
in a function. The `@function_tool` decorator builds the tool schema from the function's name, type
hints, and docstring - so **the docstring is the interface**. It is how the model decides which
retriever fits the question in front of it.

Below, three retrieval strategies with genuinely different strengths, and one agent that chooses.

In [11]:
@function_tool
async def search_news(question: str) -> str:
    """Search news article text by meaning. Best for broad themes and open-ended questions,
    for example "what are the concerns around AI regulation"."""
    result = vector_retriever.search(query_text=question, top_k=5)
    return json.dumps([item.content for item in result.items], indent=2)


@function_tool
async def search_news_with_context(question: str) -> str:
    """Search news by meaning and return graph context with each result: the source article,
    its date and sentiment, the companies mentioned, and their competitors. Use this whenever
    the answer needs citations or company relationships."""
    result = hybrid_graph_retriever.search(query_text=question, top_k=5)
    return json.dumps([json.loads(item.content) for item in result.items], indent=2)


@function_tool
async def find_named_entity(question: str) -> str:
    """Search news for an exact company name, ticker, or product that appears literally in the
    text. Use this when the question centres on a specific named thing rather than a theme."""
    result = hybrid_retriever.search(query_text=question, top_k=5)
    return json.dumps([item.content for item in result.items], indent=2)


RETRIEVAL_PROMPT = """You answer questions about companies using a Neo4j news graph.

Choose the retrieval tool that fits the question:
- search_news for broad themes and open-ended topics.
- search_news_with_context when the answer needs citations, dates, sentiment, or company
  relationships. Prefer this one for anything analytical.
- find_named_entity when the question is about a specific named company, ticker, or product.

Cite the article title and date when the tool provides them. If the retrieved passages do not
answer the question, say so rather than filling the gap from your own knowledge."""


async def run_retrieval_agent(query: str):
    agent = Agent(
        name="news_analyst",
        instructions=RETRIEVAL_PROMPT,
        tools=[search_news, search_news_with_context, find_named_entity],
        model=MODEL,
    )
    print(f"Query: {query}\n")
    result = await Runner.run(agent, query)
    print(f"Result: {result.final_output}")
    return result


print("Retrieval agent defined ✓")

Retrieval agent defined ✓


In [13]:
await run_retrieval_agent(
    "Which companies are working in data centre space, and who competes with them?"
)

Query: Which companies are working in data centre space, and who competes with them?

Result: Companies in the data centre space mentioned here include:

- **Iron Mountain** — data center and data storage provider; competes with other co-location/data center operators, though no direct competitors are listed in the result.
- **vXchnge** — data center operator; no competitors listed in the result.
- **Canberra Data Centres** — co-location/data centre provider; competitors listed: **Australian Data Centres**.
- **Microsoft Corporation** — active in cloud/data centre infrastructure; competitors listed in this result include **Apple**, **Netflix, Inc.**, and others, but these are broader product/market competitors rather than pure data centre peers.

Related companies in the broader space:
- **Sparkle**
- **PacketFabric**
- **Web Werks** (via Iron Mountain JV)
- **Chorus** (co-location facilities mentioned)

Source: *Making the most of the new data centre and co-location landscape* (2022-0

RunResult(input='Which companies are working in data centre space, and who competes with them?', new_items=[ToolCallItem(agent=Agent(name='news_analyst', handoff_description=None, tools=[FunctionTool(name='search_news', description='Search news article text by meaning. Best for broad themes and open-ended questions,\nfor example "what are the concerns around AI regulation".', params_json_schema={'properties': {'question': {'title': 'Question', 'type': 'string'}}, 'required': ['question'], 'title': 'search_news_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x75688e1b17f0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None), FunctionTool(name='search_news_

In [14]:
await run_retrieval_agent("What are the recent trends in cloud computing?")

Query: What are the recent trends in cloud computing?

Result: Recent cloud-computing trends in the retrieved news are:

- **Strong market growth / adoption:** Cloud use is rising and being described as a **strategic imperative** for IT rather than a nice-to-have.  
  - *“Cloud Services Is No Longer a Nice-to-Have, But Necessity for IT world”* — **2023-02-23**
- **Broadening enterprise adoption of SaaS and hosted infrastructure:** Organizations of all sizes are increasingly using **SaaS** and hosted cloud services.  
  - *“ComputerSupport.com says cloud computing on the rise”* — **2014-05-01**
- **Focus on efficiency, cost savings, and security:** Cloud is being positioned as a way to improve **productivity, speed, performance, and security**.  
  - *“Cloud Computing Solutions Market Expects +16% CAGR Growth…”* — **2019-12-25**
- **More integration/partnerships:** Cloud providers are forming partnerships to integrate more functions across platforms.  
  - *“Will cloud computing transfo

RunResult(input='What are the recent trends in cloud computing?', new_items=[ToolCallItem(agent=Agent(name='news_analyst', handoff_description=None, tools=[FunctionTool(name='search_news', description='Search news article text by meaning. Best for broad themes and open-ended questions,\nfor example "what are the concerns around AI regulation".', params_json_schema={'properties': {'question': {'title': 'Question', 'type': 'string'}}, 'required': ['question'], 'title': 'search_news_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x75688e1b17f0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None), FunctionTool(name='search_news_with_context', description='Sea

Nothing routes those calls but the docstrings - no keyword matching, no conditional logic. Rewriting a
description is the fastest way to correct a retriever that is being over- or under-used.

Inspect `result.new_items` to see exactly which tool ran:

In [15]:
result = await run_retrieval_agent("What is being reported about Nvidia GPU supply?")

print("\nTool calls made:")
for item in result.new_items:
    name = getattr(getattr(item, "raw_item", None), "name", None)
    if name:
        print(f"  - {name}")

Query: What is being reported about Nvidia GPU supply?

Result: Reports suggest Nvidia is **not aggressively boosting RTX 40-series GPU supply** yet.  
Specifically, one report says Nvidia is **letting partners clear remaining RTX 30-series inventory** before ramping up the newer cards, and its packaging/testing suppliers **haven’t seen increased orders**.

Source: **“Nvidia Reportedly in No Rush to Boost RTX 40-Series Output”** — **2023-04-14**

Tool calls made:
  - search_news_with_context


## 8. Specialist agents and handoffs

One agent with three tools works well. The Agents SDK offers a second pattern that suits retrieval
just as naturally: **handoffs**, where an agent transfers the conversation to another agent entirely.

The difference matters when specialists need their own instructions. A triage agent with a short
prompt routes the question; each specialist carries only the prompt and the single tool it needs, so
neither one is reasoning about tools it will never call.

Handoffs appear to the model as tools named `transfer_to_<agent_name>`, so routing stays a model
decision - but once transferred, the specialist owns the rest of the turn.

In [16]:
theme_specialist = Agent(
    name="theme_specialist",
    handoff_description="Handles broad, open-ended questions about topics and trends.",
    instructions=(
        "You answer broad questions about industry themes using `search_news`. "
        "Summarise what the retrieved passages say. Do not add outside knowledge."
    ),
    tools=[search_news],
    model=MODEL,
)

analysis_specialist = Agent(
    name="analysis_specialist",
    handoff_description="Handles analytical questions needing citations, dates, or company relationships.",
    instructions=(
        "You answer analytical questions using `search_news_with_context`, which returns the "
        "source article, its date and sentiment, and related companies. Always cite the article "
        "title and date. Do not add outside knowledge."
    ),
    tools=[search_news_with_context],
    model=MODEL,
)

entity_specialist = Agent(
    name="entity_specialist",
    handoff_description="Handles questions about a specific named company, ticker, or product.",
    instructions=(
        "You answer questions about specific named companies or products using `find_named_entity`, "
        "which matches exact terms in the article text. Do not add outside knowledge."
    ),
    tools=[find_named_entity],
    model=MODEL,
)

triage_agent = Agent(
    name="retrieval_triage",
    instructions=(
        "You route news questions to the right specialist. Read the question, pick one specialist, "
        "and hand off. Do not answer the question yourself."
    ),
    handoffs=[theme_specialist, analysis_specialist, entity_specialist],
    model=MODEL,
)

print("Triage agent and 3 specialists defined ✓")

Triage agent and 3 specialists defined ✓


In [17]:
query = "How has sentiment around semiconductor manufacturing changed recently?"
print(f"Query: {query}\n")

result = await Runner.run(triage_agent, query)

print(f"Handled by: {result.last_agent.name}")
print(f"\nResult: {result.final_output}")

Query: How has sentiment around semiconductor manufacturing changed recently?

Handled by: theme_specialist

Result: Sentiment has turned more cautious recently.

The retrieved passages say:
- Demand and equipment spending were very strong, but signs of a cyclical downturn are now emerging.
- Some companies warned of weaker second-half sales, citing inflation and softer spending on phones, PCs, servers, and data centers.
- Others still sounded constructive on 5G and data center demand, but with less confidence about broad market growth.
- Order cuts, inventory reduction, and slowing growth rates suggest the industry is cooling after a very strong run.

So overall: from highly upbeat to mixed-to-negative, with downturn concerns now more visible.


`result.last_agent` shows which specialist finished the turn. Because handoffs are just tools under
the hood, the same `Runner.run()` call covers the whole chain, and the SDK's built-in tracing records
every hop.

Use tools when one agent should stay in charge and combine several sources. Use handoffs when the
specialists need materially different instructions, or when you want the routing step visible and
auditable on its own.

## 9. Comparing strategies side by side

Tool choice is the agent's problem. Knowing which retriever suits *your* data is yours. Run all three
over one question and read the results together.

In [18]:
COMPARISON_QUERY = "semiconductor manufacturing capacity"

for label, retriever in [
    ("vector", vector_retriever),
    ("hybrid", hybrid_retriever),
    ("hybrid + graph", hybrid_graph_retriever),
]:
    print(f"\n=== {label} ===")
    for item in retriever.search(query_text=COMPARISON_QUERY, top_k=2).items:
        try:
            # Retrievers with a result_formatter return JSON; the others return
            # the node's string repr.
            row = json.loads(item.content)
            print(f"  [{row['title']}] {row['text'][:130].strip()}...")
        except (json.JSONDecodeError, KeyError):
            print(f"  {item.content[:150].strip()}...")


=== vector ===
  {'text': ', therefore, demand for semiconductor production equipment – is extremely sensitive to market conditions.\nWhen they deteriorate, companies...
  {'text': 'Demand for semiconductors and the equipment used in their production has been sky high in recent years, an up-and-up trend industry forecast...

=== hybrid ===
  {'text': ', therefore, demand for semiconductor production equipment – is extremely sensitive to market conditions.\nWhen they deteriorate, companies...
  {'text': "Following a DIGITIMES Asia report in mid-May indicating that Chinese foundry SMIC has removed its 14nm FinFET offering from its website, rec...

=== hybrid + graph ===
  [Semiconductor cycle shows signs of peaking] , therefore, demand for semiconductor production equipment – is extremely sensitive to market conditions.
When they deteriorate, c...
  [SMIC subsidiary continues to fulfill Chinese demands for FinFET capabilities] Following a DIGITIMES Asia report in mid-May indicating that

Judge them on whether the retrieved passages would let a model answer your questions: recall across
the topic, precision on named entities, and whether the extra graph context justifies the extra Cypher.

In [19]:
driver.close()
print("Driver closed ✓")

Driver closed ✓


## 10. Summary

| Pattern | Key APIs | Use Case |
|---------|----------|----------|
| Retrievers as tools | `@function_tool`, `Agent(tools=[...])` | One agent picks a retrieval strategy per question |
| Specialist handoffs | `Agent(handoffs=[...])`, `result.last_agent` | Routing is its own step; specialists carry their own prompts |
| Graph traversal retrieval | `VectorCypherRetriever`, `HybridCypherRetriever` | Retrieved text arrives with citations and relationships |

### Key Implementation Notes

- **Index pairing** - the embedding model must match the model that built the vector index. A mismatch does not raise an error, it silently returns meaningless results. Re-embed a stored chunk and compare against its saved vector to confirm.
- **`retrieval_query`** - `node` and `score` are in scope, so retrieval can continue into the graph from each vector hit. This is what makes results citable.
- **`result_formatter`** - shapes each record into whatever the model should see. Without it, results arrive as the driver's default record repr.
- **Docstrings are the tool interface** - `@function_tool` builds the schema from the function's name, hints, and docstring, so the description is what drives tool selection.
- **Cypher 25** - `neo4j-graphrag` emits the `SEARCH` clause; databases that still default to Cypher 5 need the generated queries prefixed.

### Resources

- [OpenAI Agents SDK Documentation](https://openai.github.io/openai-agents-python/)
- [OpenAI Agents SDK - Handoffs](https://openai.github.io/openai-agents-python/handoffs/)
- [neo4j-graphrag for Python](https://neo4j.com/docs/neo4j-graphrag-python/current/)
- [neo4j-graphrag - RAG user guide](https://neo4j.com/docs/neo4j-graphrag-python/current/user_guide_rag.html)
- [Neo4j Python Driver Documentation](https://neo4j.com/docs/python-manual/current/)